In [3]:
import torch
import torch.nn as nn

In [7]:
"""
Détails du GRU :
    - embedding_dim (input_size) : Le nombre de features d'une entrée, dans notre cas ce sera la dimension
    des embeddings,
    - hidden_size : La taille du vecteur d'état (le nombre de neurones récurrents),
    - num_layers : Le nombre de réseaux GRU à empiler,
    - vocab_size (num_classes dans la documentation) : Dans notre cas sera la taille du 
    vocabulaire, sela servira à construire
    la distribution de probabilité sur le vocabulaire à partir d'un vecteur d'état

    * batch_first = True : On aura donc des données organiséses dans un tenseur comme suit :
     (B, S, F)
        B : Numéro du batch
        S : Numéro de la séquence (au sein du batch) ==> nombre de lignes
        F : Numéro du feature (au sein de la séquence) ==> nombre de colonnes
"""

"\nDétails du GRU :\n    - embedding_dim (input_size) : Le nombre de features d'une entrée, dans notre cas ce sera la dimension\n    des embeddings,\n    - hidden_size : La taille du vecteur d'état (le nombre de neurones récurrents),\n    - num_layers : Le nombre de réseaux GRU à empiler,\n    - vocab_size (num_classes dans la documentation) : Dans notre cas sera la taille du \n    vocabulaire, sela servira à construire\n    la distribution de probabilité sur le vocabulaire à partir d'un vecteur d'état\n\n    * batch_first = True : On aura donc des données organiséses dans un tenseur comme suit :\n     (B, S, F)\n        B : Numéro du batch\n        S : Numéro de la séquence (au sein du batch) ==> nombre de lignes\n        F : Numéro du feature (au sein de la séquence) ==> nombre de colonnes\n"

In [8]:
class GRU(nn.Module):
    def __init__(self, embedding_dim, vocab_size, hidden_size, num_layers):
        super(GRU, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(
            input_size=embedding_dim, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True
        )

        self.probs = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x) # x : (batch, seq, embedding_dim)
        output, h_n = self.gru(x)

        logits = self.probs(output)
        return logits, h_n



## Tests (Pour voir si les dimensions et autres collent)

In [9]:
# test 
embedding_dim = 10
hidden_size = 5
num_layers = 1
vocab_size = 20
 
model = GRU(embedding_dim, vocab_size, hidden_size, num_layers)
print(model)

x = torch.tensor([
    [1, 5, 3],
    [3, 2, 2]
])

logits, h_n = model.forward(x)
print(f"Shape des logits : ", logits.shape)
for b in range(logits.shape[0]) :
    print(f"Batch n{b} :")
    for line in range(logits.shape[1]):
        print(logits[b, line])

print(f"Shape du vecteur d'état : ", h_n.shape)
for b in range(h_n.shape[0]) :
    print(f"Batch n{b} :")
    for line in range(h_n.shape[1]):
        print(h_n[b, line])


GRU(
  (embedding): Embedding(20, 10)
  (gru): GRU(10, 5, batch_first=True)
  (probs): Linear(in_features=5, out_features=20, bias=True)
)
Shape des logits :  torch.Size([2, 3, 20])
Batch n0 :
tensor([-0.2928,  0.4132,  0.0479,  0.4513,  0.6099, -0.2403,  0.4068,  0.0250,
        -0.0558, -0.3371, -0.3094, -0.3399, -0.5099,  0.3666,  0.1902,  0.0601,
         0.1424,  0.5228, -0.0345,  0.0486], grad_fn=<SelectBackward0>)
tensor([-0.3185,  0.4259,  0.5201,  0.3397,  0.3644, -0.7955,  0.4688,  0.1830,
        -0.5961, -0.0567, -0.5467, -0.5079, -0.0576,  0.6181,  0.0967,  0.1492,
         0.1888,  0.6092,  0.2283, -0.0767], grad_fn=<SelectBackward0>)
tensor([-0.3095,  0.4164,  0.6282,  0.3398,  0.3012, -0.9501,  0.4511,  0.1766,
        -0.7540,  0.0014, -0.5946, -0.5784,  0.0578,  0.7632,  0.0674,  0.1409,
         0.1696,  0.6890,  0.2636, -0.1520], grad_fn=<SelectBackward0>)
Batch n1 :
tensor([-0.2391,  0.4083,  0.6252,  0.2963,  0.2778, -0.8105,  0.4907,  0.3702,
        -0.6626, -0.

In [10]:
# Nous appliquerons un softmax sur les logits, pour avoir une distribution de probabilités
softmax = nn.Softmax(dim=1)
probs = softmax(logits)

print(f"Shape des probs : ", probs.shape)
for b in range(probs.shape[0]) :
    print(f"Batch n{b} :")
    for line in range(probs.shape[1]):
        print(probs[b, line])

Shape des probs :  torch.Size([2, 3, 20])
Batch n0 :
tensor([0.3381, 0.3316, 0.2278, 0.3586, 0.3973, 0.4841, 0.3216, 0.2999, 0.4808,
        0.2683, 0.3936, 0.3798, 0.2306, 0.2651, 0.3578, 0.3147, 0.3252, 0.3057,
        0.2741, 0.3703], grad_fn=<SelectBackward0>)
tensor([0.3295, 0.3358, 0.3653, 0.3207, 0.3108, 0.2779, 0.3422, 0.3512, 0.2801,
        0.3552, 0.3104, 0.3211, 0.3625, 0.3409, 0.3258, 0.3441, 0.3406, 0.3333,
        0.3565, 0.3267], grad_fn=<SelectBackward0>)
tensor([0.3325, 0.3326, 0.4070, 0.3207, 0.2918, 0.2381, 0.3362, 0.3489, 0.2392,
        0.3764, 0.2959, 0.2992, 0.4069, 0.3941, 0.3164, 0.3412, 0.3342, 0.3610,
        0.3693, 0.3030], grad_fn=<SelectBackward0>)
Batch n1 :
tensor([0.3279, 0.3125, 0.4601, 0.2338, 0.2704, 0.2068, 0.4003, 0.4207, 0.2177,
        0.4526, 0.2271, 0.3033, 0.4427, 0.3332, 0.3178, 0.4285, 0.3474, 0.3013,
        0.4012, 0.3055], grad_fn=<SelectBackward0>)
tensor([0.3393, 0.3303, 0.2940, 0.3500, 0.3405, 0.3471, 0.3080, 0.3057, 0.3440,
        

## Problème de phrases de longueur différentes : utilisation de padding 

### Faire une petite analyse du corpus, pour connaitre les longueur des phrases

In [12]:
corpus_filename = "../../data/corpus/corpus_complet_12k.bpe"
length_dict = {}

with open(corpus_filename, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()  # enlève le \n à la fin
        
        length = len(line)

        if length in length_dict:
            length_dict[length] += 1
        else:
            length_dict[length] = 1

print(length_dict)

{102: 2095, 319: 233, 449: 67, 370: 118, 166: 1106, 207: 709, 161: 1110, 299: 248, 360: 133, 283: 318, 217: 622, 87: 2411, 279: 308, 233: 507, 340: 159, 222: 609, 355: 149, 1106: 3, 277: 350, 435: 76, 377: 118, 374: 123, 485: 34, 243: 456, 188: 895, 670: 8, 255: 440, 474: 56, 235: 536, 258: 406, 195: 803, 149: 1339, 224: 609, 202: 735, 354: 145, 302: 248, 81: 2699, 383: 97, 499: 46, 86: 2453, 492: 47, 247: 442, 199: 771, 191: 802, 303: 263, 446: 48, 151: 1337, 341: 168, 471: 60, 291: 287, 386: 100, 194: 806, 385: 111, 76: 2737, 131: 1529, 62: 3165, 523: 28, 364: 120, 1540: 1, 580: 35, 345: 143, 343: 149, 743: 9, 601: 15, 15: 4990, 19: 3029, 154: 1297, 156: 1198, 26: 3325, 147: 1340, 141: 1399, 73: 2872, 46: 3685, 24: 3284, 155: 1201, 150: 1312, 169: 1078, 173: 1025, 58: 3389, 13: 4347, 38: 3614, 29: 3602, 75: 2739, 88: 2510, 39: 3538, 31: 3525, 208: 728, 47: 3677, 124: 1753, 152: 1326, 65: 3136, 48: 3672, 35: 3492, 25: 3210, 56: 3454, 85: 2381, 27: 3741, 126: 1660, 80: 2651, 82: 2543, 

In [ ]:
import pandas as pd

df_lengths = pd.DataFrame({
    "longueur_phrase": list(length_dict.keys()),
    "nombre_phrases": list(length_dict.values())
})

df_lengths = df_lengths.sort_values("longueur_phrase").reset_index(drop=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

df_lengths

## Test pour voir comment les phrases seront paddées

In [3]:
raw_batches = []
current_batch = []
current_min_len = None
batch_size = 1

sentences = [
    torch.tensor([1, 2, 3, 4, 5]),
    torch.tensor([1, 2, 3]),
    torch.tensor([1, 2, 3, 4, 5, 4, 6, 8, 9 ,0]),
    torch.tensor([1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5]),
    
]
for sent in sentences:
    sent_len = len(sent)

    if len(current_batch) == 0:
        current_batch = [sent]
        current_min_len = sent_len

    elif len(current_batch) < batch_size and sent_len - current_min_len <= 5:
        current_batch.append(sent)

    else:
        raw_batches.append(current_batch)
        current_batch = [sent]
        current_min_len = sent_len

    if len(current_batch) > 0:
        raw_batches.append(current_batch)


    print(raw_batches)

[[tensor([1, 2, 3, 4, 5])]]
[[tensor([1, 2, 3, 4, 5])], [tensor([1, 2, 3, 4, 5])], [tensor([1, 2, 3])]]
[[tensor([1, 2, 3, 4, 5])], [tensor([1, 2, 3, 4, 5])], [tensor([1, 2, 3])], [tensor([1, 2, 3])], [tensor([1, 2, 3, 4, 5, 4, 6, 8, 9, 0])]]
[[tensor([1, 2, 3, 4, 5])], [tensor([1, 2, 3, 4, 5])], [tensor([1, 2, 3])], [tensor([1, 2, 3])], [tensor([1, 2, 3, 4, 5, 4, 6, 8, 9, 0])], [tensor([1, 2, 3, 4, 5, 4, 6, 8, 9, 0])], [tensor([1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5])]]


## Test complet sur la fonction qui fera les paddings des phrases 

In [ ]:
import random
from torch.nn.utils.rnn import pad_sequence
def pad_sequences(
        filename,
        token_to_id,
        batch_size,
        max_diff=5,
        pad_token="<pad>",
        unk_token="<unk>",
        shuffle_batches=True
):
    
    pad_id = token_to_id[pad_token]
    unk_id = token_to_id[unk_token]

    sentences = []

    # Lire le fichier et transformer chaque phrase en un tenseur 
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            tokens = line.strip().split()
            if len(tokens) == 0 : continue

            ids = [token_to_id.get(tok, unk_id) for tok in tokens]

            sentence_tensor = torch.tensor(ids)
            sentences.append(sentence_tensor)

    # Trier les phrases par longueur 
    sentences = sorted(sentences, key=lambda x: len(x))

    # Creer les groupes de phrases dont la différence de longueur <= max_diff
    raw_batches = []
    current_batch = []
    current_min_len = None

    for sent in sentences:
        sent_len = len(sent)

        if len(current_batch) == 0:
            current_batch = [sent]
            current_min_len = sent_len

        elif len(current_batch) < batch_size and sent_len - current_min_len <= max_diff:
            current_batch.append(sent)

        else:
            raw_batches.append(current_batch)
            current_batch = [sent]
            current_min_len = sent_len

    if len(current_batch) > 0:
        raw_batches.append(current_batch)

    if shuffle_batches:
        random.shuffle(raw_batches)

    # Ajouter pading pour chaque batch
    padded_batches = []
    for batch in raw_batches :
        lengths = torch.tensor([len(sent) for sent in batch], dtype=torch.long)

        batch_padded = pad_sequence(
            batch,
            batch_first=True,
            padding_value=pad_id
        )

        padded_batches.append((batch_padded, lengths))

    return padded_batches


In [ ]:
file = "test.txt"
tokenizer = "../../data/bpe_embeddings/bpe_tokenizer_vocab_12k.json"
import json

with open(tokenizer, 'r') as f:
    data = json.load(f)

tok = data["model"]["vocab"]
batches = pad_sequences(
        file,
        tok,
        batch_size=2,
        max_diff=5,
        pad_token="<pad>",
        unk_token="<unk>",
        shuffle_batches=True
)

In [ ]:
print(batches)

## Comment ce padding affecte le forward, et comment on moyenne les loss du train et du val lors du train

In [4]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class GRU(nn.Module):
    def __init__(self, embedding_dim, vocab_size, hidden_size, num_layers):
        super(GRU, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=embedding_dim, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True
        )

        self.probs = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, lengths):
        print("=== DANS FORWARD ===")
        print("x au départ :")
        print(x)
        print("x.shape :", x.shape)
        print("lengths :", lengths)
        print()

        # 1. Embedding
        x = self.embedding(x)
        print("Après embedding :")
        print("x.shape :", x.shape)
        print()

        # 2. Pack
        x_packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        print("Après pack_padded_sequence :")
        print("x_packed.data.shape :", x_packed.data.shape)
        print("x_packed.batch_sizes :", x_packed.batch_sizes)
        print()

        # 3. GRU
        output_packed, h_n = self.gru(x_packed)

        print("Après GRU :")
        print("output_packed.data.shape :", output_packed.data.shape)
        print("h_n.shape :", h_n.shape)
        print()

        # 4. Dépack
        output, output_lengths = pad_packed_sequence(
            output_packed,
            batch_first=True
        )

        print("Après pad_packed_sequence :")
        print("output.shape :", output.shape)
        print("output_lengths :", output_lengths)
        print()

        # 5. Projection vers le vocabulaire
        logits = self.probs(output)

        print("Après Linear self.probs :")
        print("logits.shape :", logits.shape)
        print()

        return logits, h_n

In [5]:
embedding_dim = 4
vocab_size = 10
hidden_size = 6
num_layers = 1

model = GRU(
    embedding_dim=embedding_dim,
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    num_layers=num_layers
)

In [6]:
pad_id = 0

batch = torch.tensor([
    [2, 3, 4, 5, 1],   # phrase de longueur 5
    [6, 7, 1, 0, 0]    # phrase de longueur 3 + padding
])

lengths = torch.tensor([5, 3])

In [7]:
x = batch[:, :-1]
y = batch[:, 1:]

lengths_x = lengths - 1

print("=== AVANT FORWARD ===")
print("batch :")
print(batch)
print("batch.shape :", batch.shape)
print()

print("x :")
print(x)
print("x.shape :", x.shape)
print()

print("y :")
print(y)
print("y.shape :", y.shape)
print()

print("lengths :", lengths)
print("lengths_x :", lengths_x)
print()

=== AVANT FORWARD ===
batch :
tensor([[2, 3, 4, 5, 1],
        [6, 7, 1, 0, 0]])
batch.shape : torch.Size([2, 5])

x :
tensor([[2, 3, 4, 5],
        [6, 7, 1, 0]])
x.shape : torch.Size([2, 4])

y :
tensor([[3, 4, 5, 1],
        [7, 1, 0, 0]])
y.shape : torch.Size([2, 4])

lengths : tensor([5, 3])
lengths_x : tensor([4, 2])



In [10]:
logits, h_n = model(x, lengths_x)

print("=== SORTIE FINALE ===")
print("logits.shape :", logits.shape)
print("h_n.shape :", h_n.shape)
print("Dernier état caché de la  1ere phrase : ", h_n[0,0].shape)
print("Dernier état caché de la  2ere phrase : ", h_n[0,1].shape)

=== DANS FORWARD ===
x au départ :
tensor([[2, 3, 4, 5],
        [6, 7, 1, 0]])
x.shape : torch.Size([2, 4])
lengths : tensor([4, 2])

Après embedding :
x.shape : torch.Size([2, 4, 4])

Après pack_padded_sequence :
x_packed.data.shape : torch.Size([6, 4])
x_packed.batch_sizes : tensor([2, 2, 1, 1])

Après GRU :
output_packed.data.shape : torch.Size([6, 6])
h_n.shape : torch.Size([1, 2, 6])

Après pad_packed_sequence :
output.shape : torch.Size([2, 4, 6])
output_lengths : tensor([4, 2])

Après Linear self.probs :
logits.shape : torch.Size([2, 4, 10])

=== SORTIE FINALE ===
logits.shape : torch.Size([2, 4, 10])
h_n.shape : torch.Size([1, 2, 6])
Dernier état caché de la  1ere phrase :  torch.Size([6])
Dernier état caché de la  2ere phrase :  torch.Size([6])


In [16]:
reshaped_logits = logits.reshape(-1, logits.size(-1))
y_reshaped = y.reshape(-1)
print("logits.shape :", logits.shape)
print("Reshaped logits = ", reshaped_logits.shape)

print("Score (pour chaque mot du vocabulaire) pour prédire le token suivant à la position 2 dans la phrase 0 = \n", logits[0,2])

print("y.shape :", y.shape)
print("y reshaped :", y_reshaped.shape)

logits.shape : torch.Size([2, 4, 10])
Reshaped logits =  torch.Size([8, 10])
Score (pour chaque mot du vocabulaire) pour prédire le token suivant à la position 2 dans la phrase 0 = 
 tensor([-0.3673, -0.2216, -0.0833,  0.2011,  0.4205, -0.2921,  0.0275,  0.3296,
         0.1939, -0.1891], grad_fn=<SelectBackward0>)
y.shape : torch.Size([2, 4])
y reshaped : torch.Size([8])


In [ ]:
"""
EXPLICATION :
    x : represente la partie des prefixes (l'historique), qui va être appris lors de
    lentraînement.
    y : ce ont les targets repsectifs de chaque mot du prefixes.
        batch = [
                    [2, 3, 4, 5, 1],
                    [6, 7, 1, 0, 0]
                ]
        x prend en compte tout sauf le dernier mot et y prend tout sauf le premier

    - On se retrouve don c avec  ces longuers pour les phrases de x : lengths_x = tensor([4, 2])

    - Après embedding :
        x.shape : torch.Size([2, 4, 4])
        On a 2 phrases, de taille 4 tokens (paddings inclus), chaque token à un embedding de 
        dimension 4

    - Après pack_padded_sequence :
    Preuve que avec "pack_padded_sequence" les paddings ne seront pas pris en compte et c'est 
    ce qu'on veut

        x_packed.data.shape : torch.Size([6, 4])
        - Avec les tokens réels de chaque phrase (4 et 2), nous obtenons 6 tokens en tout
        - Chaque token a un embedding de dimension 4

        x_packed.batch_sizes : tensor([2, 2, 1, 1]) 
        Montre combien de phrases sont 'actives' à chaque pas (à chaque fois qu'on avance d'un mot)
        et c'est pour ça qu'on voit le changement dans la taille du batch (qui passe de 2 phrases à 1)
                phrase1     phrase2
        t=0     2           6      : 2 phrases actives
        t=1     3           7      : 2 phrases actives
        t=2     4                  : 1 phrase active
        t=3     5                  : 1 phrase active


    - Après GRU :
        output_packed.data.shape : torch.Size([6, 6])
        - Nous avons 6 tokens réels à traiter avec un état caché dont la taille est de 6

        h_n.shape : torch.Size([1, 2, 6])
        - Dimension du dernier état caché : 1 batch, 2 phrases et taille 6 en état caché

    
    - Après Linear self.probs :
        logits.shape : torch.Size([2, 4, 10])
        Après projection sur le vocabulaire : 2 phrases, 4 tokens chacune, 
        et logit sur 10 tokens du vocabulaire


    - les logits : un score (pour chque mot du vocabulaire) pour prédire le mot suivant sur 
    toutes les positions pour toutes les phrases
    SAUF QUE : 
        - L'entropie croisée s'attend à avoir quelque chose comme :
            prédictions (logits) : (N, C) 
            cibles (tagets) : (N)

            avec :
                N : nombre total de prédiction
                C : nombre de classes (taille du vocabulaire)
        C'est pour cela qu'avant de passer les logits et les targets à la fonction de perte 
        on fait une reshape

        On obtient : 
            Reshaped logits =  torch.Size([8, 10])
            y reshaped : torch.Size([8])
"""

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    batches,
    batch_size=1,
    shuffle=True,
    collate_fn=lambda batch: batch[0]
)

print(type(loader))


In [ ]:

for batch, lengths in loader :
    i = 0
    print("Shape du batch : ", batch.shape)
    #print(lengths.shape)

    print("length = ", lengths)
    print("batch : ", batch)
    x = batch[:, :lengths[i]]
    i+=1
    print("x = ", x)



## Test d'entraînement complet

In [35]:
import importlib
import hp
import gru
import utils
import Vocab
from torch.utils.data import DataLoader
importlib.reload(hp)
importlib.reload(gru)
importlib.reload(utils)
importlib.reload(Vocab)
from hp  import HP_GRU
from gru import GRU
from Vocab import Vocab
from utils import pad_sequences
import json

In [19]:
config_file = "configs/config_1.json"
hp = HP_GRU(config_file)
model = GRU(hp)

train_file = model.hp.train_filename
val_file = model.hp.val_filename
vocab_filename = model.hp.vocab_filename

with open(vocab_filename, 'r') as f:
    data = json.load(f)

tok = data["model"]["vocab"]

train_batches = pad_sequences(
        train_file,
        tok,
        batch_size=model.hp.minibatch,
        max_diff=5,
        pad_token="<pad>",
        unk_token="<unk>",
        shuffle_batches=True
)
val_batches = pad_sequences(
        val_file,
        tok,
        batch_size=model.hp.minibatch,
        max_diff=5,
        pad_token="<pad>",
        unk_token="<unk>",
        shuffle_batches=True
)

train_loader = DataLoader(
    train_batches,
    batch_size=model.hp.minibatch,
    shuffle=True,
    collate_fn=lambda batch: batch[0]
)

val_loader = DataLoader(
    val_batches,
    batch_size=model.hp.minibatch,
    shuffle=True,
    collate_fn=lambda batch: batch[0]
)

vocab size (depuis fichier global) = 12001 emb_dim = 32
Taille du vocabulaire : 12001
Indice de <pad> : 12000
Indice de <unk> : 0
Taille embeddings : torch.Size([12001, 32])


In [20]:
exe_time, best_metrics = model.fit(train_loader, val_loader, 5, 1, lr=1e-3, patience=2)

Époque 1/5 | Train Loss: 0.3479 | Val Loss: 0.0999
 --> Nouveau best metrics détecté
Époque 2/5 | Train Loss: 0.0958 | Val Loss: 0.0998
 --> Nouveau best metrics détecté
Époque 3/5 | Train Loss: 0.1706 | Val Loss: 1.3350
 --> Stagnation détectée. Patience : 1/2
Époque 4/5 | Train Loss: 0.1699 | Val Loss: 1.3343
 --> Stagnation détectée. Patience : 2/2

[Early Stopping] Pas d'amélioration de la val loss depuis 2 époques. Arrêt de l'entraînement à l'époque 4.
-> Fichier .pt du Best Model enregistré avec succès.


In [2]:
config_file = "configs/config_2.json"

hp = HP_GRU(config_file)
model = GRU(hp)

train_file = model.hp.train_filename
val_file = model.hp.val_filename

# Vocabulaire construit lors de la lecture du fichier d'embeddings
tok = model.vocab.dico_voca

print("Taille du vocabulaire :", model.vocab.vocab_size)
print("Taille de la matrice :", model.vocab.matrice.shape)
print("ID de <unk> :", model.vocab.unk_id)
print("ID de <pad> :", model.vocab.pad_id)

assert tok["<pad>"] == model.pad_id
assert tok["<unk>"] == model.vocab.unk_id
assert model.vocab.vocab_size == model.vocab.matrice.shape[0]

strategy = 2
emb_filename = ../../data/bpe_embeddings/bpe_embeddings_12k_32.txt
Taille annoncée dans le fichier : 11844
Tokens réellement lus : 11844
Taille finale du vocabulaire : 11846
unk_id : 11844
pad_id : 11845
Forme de la matrice : torch.Size([11846, 32])
Taille du vocabulaire : 11846
Indice de <pad> : 11845
Indice de <unk> : 11844
Taille embeddings : torch.Size([11846, 32])
Taille du vocabulaire : 11846
Taille de la matrice : torch.Size([11846, 32])
ID de <unk> : 11844
ID de <pad> : 11845


In [5]:
train_batches = pad_sequences(
    train_file,
    tok,
    batch_size=model.hp.minibatch,
    max_diff=5,
    pad_token="<pad>",
    unk_token="<unk>",
    shuffle_batches=True
)

val_batches = pad_sequences(
    val_file,
    tok,
    batch_size=model.hp.minibatch,
    max_diff=5,
    pad_token="<pad>",
    unk_token="<unk>",
    shuffle_batches=False
)

train_loader = DataLoader(
    train_batches,
    batch_size=model.hp.minibatch,
    shuffle=True,
    collate_fn=lambda batch: batch[0]
)

val_loader = DataLoader(
    val_batches,
    batch_size=model.hp.minibatch,
    shuffle=False,
    collate_fn=lambda batch: batch[0]
)

In [6]:
exe_time, best_metrics = model.fit(train_loader, val_loader, 5, 1, lr=1e-3, patience=2)

Époque 1/5 | Train Loss: 0.3471 | Val Loss: 1.3370
 --> Nouveau best metrics détecté
Époque 2/5 | Train Loss: 0.0957 | Val Loss: 1.3335
 --> Nouveau best metrics détecté
Époque 3/5 | Train Loss: 0.1730 | Val Loss: 1.3303
 --> Nouveau best metrics détecté
Époque 4/5 | Train Loss: 0.0942 | Val Loss: 1.3265
 --> Nouveau best metrics détecté
Époque 5/5 | Train Loss: 0.0790 | Val Loss: 1.3226
 --> Nouveau best metrics détecté
-> Fichier .pt du Best Model enregistré avec succès.


In [36]:
config_file = "configs/config_1_test.json"
hp = HP_GRU(config_file)
model = GRU(hp)

train_file = model.hp.train_filename
val_file = model.hp.val_filename
vocab_filename = model.hp.vocab_filename

with open(vocab_filename, 'r') as f:
    data = json.load(f)

tok = data["model"]["vocab"]

train_batches = pad_sequences(
        train_file,
        tok,
        batch_size=model.hp.minibatch,
        max_diff=5,
        pad_token="<pad>",
        unk_token="<unk>",
        shuffle_batches=True
)
val_batches = pad_sequences(
        val_file,
        tok,
        batch_size=model.hp.minibatch,
        max_diff=5,
        pad_token="<pad>",
        unk_token="<unk>",
        shuffle_batches=True
)

train_loader = DataLoader(
    train_batches,
    batch_size=model.hp.minibatch,
    shuffle=True,
    collate_fn=lambda batch: batch[0]
)

val_loader = DataLoader(
    val_batches,
    batch_size=model.hp.minibatch,
    shuffle=True,
    collate_fn=lambda batch: batch[0]
)

vocab size (depuis fichier global) = 12001 emb_dim = 32
Taille du vocabulaire : 12001
Indice de <pad> : 12000
Indice de <unk> : 0
Taille embeddings : torch.Size([12001, 32])


In [37]:
exe_time, best_metrics = model.fit(train_loader, val_loader, 5, 1, lr=1e-3, patience=2)

torch.Size([3, 6])
Vraies longueurs des préfixes :  tensor([3, 3, 6])
Shape x packed (prefixe packed) : torch.Size([12, 32])
Shape output_packed : torch.Size([12, 5])
Shape h_n : torch.Size([1, 3, 5])
Shape output : torch.Size([3, 6, 5])

--- INSPECTION COMPLÈTE DE OUTPUT ---
Taille globale : torch.Size([3, 6, 5])

>>> PHRASE 1 (Longueur réelle de préfixe = 3) :
  Pos 0 [MOT VALIDE] -> h_0 = [-0.0051571959629654884, -0.02713043987751007, 0.6142205595970154, -0.8528374433517456, -0.5618056058883667] (Calculé par le GRU)
  Pos 1 [MOT VALIDE] -> h_1 = [-0.11511485278606415, -0.16623258590698242, 0.8686142563819885, -0.9614338278770447, -0.7396618723869324] (Calculé par le GRU)
  Pos 2 [MOT VALIDE] -> h_2 = [0.607654333114624, -0.2685892581939697, 0.7761492729187012, -0.8721063733100891, -0.03995364159345627] (Calculé par le GRU)
  Pos 3 [PADDING]    -> Vector = [0.0, 0.0, 0.0, 0.0, 0.0] (Injecté par pad_packed_sequence, PAS de calcul !)
  Pos 4 [PADDING]    -> Vector = [0.0, 0.0, 0.0, 0.0

torch.Size([1, 45])
Vraies longueurs des préfixes :  tensor([45])
Shape x packed (prefixe packed) : torch.Size([45, 32])
Shape output_packed : torch.Size([45, 5])
Shape h_n : torch.Size([1, 1, 5])
Shape output : torch.Size([1, 45, 5])

--- INSPECTION COMPLÈTE DE OUTPUT ---
Taille globale : torch.Size([1, 45, 5])

>>> PHRASE 1 (Longueur réelle de préfixe = 45) :
  Pos 0 [MOT VALIDE] -> h_0 = [-0.3568096458911896, -0.05109637975692749, 0.2409125566482544, -0.19202429056167603, -0.5719071626663208] (Calculé par le GRU)
  Pos 1 [MOT VALIDE] -> h_1 = [-0.575461208820343, 0.3715478181838989, 0.03737368434667587, 0.07047870755195618, 0.6752054691314697] (Calculé par le GRU)
  Pos 2 [MOT VALIDE] -> h_2 = [0.3644334077835083, 0.35805249214172363, 0.24687737226486206, 0.11756138503551483, 0.7307945489883423] (Calculé par le GRU)
  Pos 3 [MOT VALIDE] -> h_3 = [0.9755719900131226, -0.16171014308929443, 0.3072805404663086, 0.4211704730987549, 0.6968337893486023] (Calculé par le GRU)
  Pos 4 [MOT VA

In [1]:
import torch


def read_corpus_cloze(
    model,
    fichier_corpus,
    vocab,
    max_lines=None,
):
    """
    Lit un corpus à trous pour un GRU.

    Pour chaque phrase :
    - récupère le mot cible ;
    - trouve sa position dans la phrase ;
    - extrait tous les mots situés avant la cible ;
    - convertit le contexte en identifiants.

    Les contextes ayant des longueurs différentes, la fonction
    retourne une liste de dictionnaires plutôt qu'un tenseur empilé.
    """

    data = []

    print("=" * 70)
    print("LECTURE DU CORPUS POUR LE GRU")
    print(f"Fichier : {fichier_corpus}")
    print("=" * 70)

    with open(fichier_corpus, "r", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            if max_lines is not None and line_number > max_lines:
                print(f"Arrêt après {max_lines} lignes.")
                break

            print("\n" + "-" * 70)
            print(f"Ligne {line_number}")
            print(f"Ligne brute : {repr(line)}")

            line = line.strip()

            if not line:
                print("Ligne vide : ignorée.")
                continue

            line_parts = line.split()

            if len(line_parts) < 2:
                print("Ligne incomplète : ignorée.")
                continue

            print(f"Éléments de la ligne : {line_parts}")

            # Le premier élément est ici considéré comme
            # l'identifiant du mot cible dans le vocabulaire.
            try:
                target_word_idx = int(line_parts[0])
            except ValueError:
                print(
                    f"Impossible de convertir '{line_parts[0]}' "
                    "en identifiant."
                )
                continue

            print(f"Identifiant du mot cible : {target_word_idx}")

            # Conversion de l'identifiant en mot
            target_word_text = vocab.get_word(target_word_idx)

            print(f"Mot cible : {target_word_text}")

            if target_word_text is None:
                print("Mot cible absent du vocabulaire : ligne ignorée.")
                continue

            # La phrase commence après l'identifiant
            sentence_tokens = line_parts[1:]

            print(f"Phrase : {sentence_tokens}")

            # Recherche de la position du mot cible
            try:
                pos_in_phrase = sentence_tokens.index(target_word_text)
            except ValueError:
                print(
                    f"Le mot cible '{target_word_text}' "
                    "n'est pas présent dans la phrase."
                )
                continue

            print(
                f"Position du mot cible dans la phrase : "
                f"{pos_in_phrase}"
            )

            # Tous les mots situés avant le mot cible
            context_tokens = sentence_tokens[:pos_in_phrase]

            print(f"Contexte gauche complet : {context_tokens}")
            print(
                f"Longueur du contexte : {len(context_tokens)}"
            )

            # Si le mot cible est le premier mot, le GRU
            # n'a aucun contexte pour effectuer une prédiction.
            if not context_tokens:
                print(
                    "Aucun mot avant la cible : ligne ignorée."
                )
                continue

            context_indices = []
            conversion_failed = False

            # Conversion des tokens du contexte en identifiants
            for token in context_tokens:

                token_idx = vocab.get_word_index2(token)

                print(
                    f"Token '{token}' -> identifiant {token_idx}"
                )

                if token_idx is None:
                    print(
                        f"Le token '{token}' est inconnu."
                    )
                    conversion_failed = True
                    break

                context_indices.append(token_idx)

            if conversion_failed:
                print(
                    "Échec de conversion du contexte : "
                    "ligne ignorée."
                )
                continue

            print(f"Indices du contexte : {context_indices}")

            # Forme du contexte : [1, longueur_sequence]
            # Le 1 correspond à la taille du batch.
            context_tensor = torch.tensor(
                [context_indices],
                dtype=torch.long,
            )

            target_tensor = torch.tensor(
                target_word_idx,
                dtype=torch.long,
            )

            print(f"Tenseur du contexte : {context_tensor}")
            print(
                f"Forme du contexte : {context_tensor.shape}"
            )
            print(f"Tenseur cible : {target_tensor}")

            data.append(
                {
                    "context": context_tensor,
                    "target": target_tensor,
                    "target_word": target_word_text,
                    "context_tokens": context_tokens,
                    "line_number": line_number,
                }
            )

            print(
                f"Exemple ajouté. Total : {len(data)}"
            )

    print("\n" + "=" * 70)
    print("FIN DE LA LECTURE")
    print(f"Nombre d'exemples valides : {len(data)}")
    print("=" * 70)

    if not data:
        return None

    return data

In [5]:
config_file = "configs/config_1_test.json"
hp = HP_GRU(config_file)
model = GRU(hp)

data_test = read_corpus_cloze(
    model=model,
    fichier_corpus="../../data/corpus/test_test.mask",
    vocab=model.vocab,
    max_lines=5,
)

print("\nRésultat retourné par la fonction :")
print(data_test)

if data_test is not None:
    print("Shape :", data_test.shape)

vocab size (depuis fichier global) = 12001 emb_dim = 32
Taille du vocabulaire : 12001
Indice de <pad> : 12000
Indice de <unk> : 0
Taille embeddings : torch.Size([12001, 32])
LECTURE DU CORPUS POUR LE GRU
Fichier : ../../data/corpus/test_test.mask

----------------------------------------------------------------------
Ligne 1
Ligne brute : "4 <s> <s> <s> vers l ' automne , malheureusement , le ménage se gâ #ta encore . </s> </s> </s>\n"
Éléments de la ligne : ['4', '<s>', '<s>', '<s>', 'vers', 'l', "'", 'automne', ',', 'malheureusement', ',', 'le', 'ménage', 'se', 'gâ', '#ta', 'encore', '.', '</s>', '</s>', '</s>']
Identifiant du mot cible : 4
Mot cible : "
Phrase : ['<s>', '<s>', '<s>', 'vers', 'l', "'", 'automne', ',', 'malheureusement', ',', 'le', 'ménage', 'se', 'gâ', '#ta', 'encore', '.', '</s>', '</s>', '</s>']
Le mot cible '"' n'est pas présent dans la phrase.

----------------------------------------------------------------------
Ligne 2
Ligne brute : '4 <s> <s> <s> le grand - p

In [6]:
# TEST EVALAUTION TEXTE A TROUS

import argparse

import torch
import torch.nn.functional as F


"""
Contrairement à l'évaluation pour le MLP, ici on ne prend pas le mot masqué entouré de k mots,
on prend toute la phrase, et tout ce qui précède le mot masqué est le contexte gauche,
et tout ce qui le suit est le contexte droit.
"""


def process_sentence(line, vocab, device, model, top_k=10):

    print("\n" + "=" * 60)
    print("Ligne reçue :", line)

    line_parts = line.split()
    print("line_parts :", line_parts)

    if not line_parts:
        return

    # Extraction du mot masqué
    masked_token_index = int(line_parts[0])
    masked_token = line_parts[masked_token_index + 1]
    masked_token_id = vocab.get_word_index(masked_token)

    print("\nPosition du mot masqué :", masked_token_index)
    print("Mot masqué :", masked_token)
    print("ID du mot masqué :", masked_token_id)

    # Extraire le contexte droit et gauche
    left_tokens = line_parts[1:masked_token_index] # exclure le numéro qui indiquait la position du mot masqué
    right_tokens = line_parts[masked_token_index + 2:]

    print("\nContexte gauche :", left_tokens)
    print("Contexte droit :", right_tokens)

    left_ids = []
    right_ids = []

    for token in left_tokens:
        token_id = vocab.get_word_index2(token)

        if token_id is None:
            return None

        left_ids.append(int(token_id))

    for token in right_tokens:
        token_id = vocab.get_word_index2(token)

        if token_id is None:
            return None

        right_ids.append(int(token_id))

    print("\nIDs du contexte gauche :", left_ids)
    print("IDs du contexte droit :", right_ids)

    # Construire le tenseur du contexte gauche pour la prédiction
    left_tensor = torch.tensor(
        [left_ids],
        dtype=torch.long,
        device=device,
    )

    print("\nTenseur du contexte gauche :", left_tensor)
    print("Shape du tenseur gauche :", left_tensor.shape)

    # Préparer le dictionnaire des probabilités
    vocab_probs = {
    token_id: {
        "token": token,
        "log-prob": 0.0
    }
    for token, token_id in vocab.dico_voca.items()
}

    print("\nNombre de candidats dans le vocabulaire :", len(vocab_probs))

    with torch.inference_mode():

        # ------------------------------------------------------
        # 1. Lire une seule fois tous les mots avant le masque
        # ------------------------------------------------------
        logits_left, hidden_left = model.forward_evaluate(left_tensor)

        print("\nAprès lecture du contexte gauche")
        print("Shape de logits_left :", logits_left.shape)
        print("Shape de hidden_left :", hidden_left.shape)

        # On extrait le dernier logit
        masked_logits = logits_left[0, -1, :]

        print("Shape de masked_logits :", masked_logits.shape)

        log_prob_masked = F.log_softmax(
            masked_logits,
            dim=-1,
        )

        print(
            "Log-probabilité initiale du vrai mot masqué :",
            log_prob_masked[masked_token_id].item()
        )

        # ------------------------------------------------------
        # 2. Ajouter P(candidat | contexte gauche)
        # ------------------------------------------------------
        for candidate_id in vocab_probs:
            vocab_probs[candidate_id]["log-prob"] += (
                log_prob_masked[candidate_id].item()
            )

        print(
            "Score initial du vrai mot masqué :",
            vocab_probs[masked_token_id]["log-prob"]
        )

        # ------------------------------------------------------
        # 3. Tester chaque candidat à la place du mot masqué
        # ------------------------------------------------------
        for candidate_id in vocab_probs:

            # Chaque candidat repart du même état caché
            hidden = hidden_left.clone()

            current_token = torch.tensor(
                [[candidate_id]],
                dtype=torch.long,
                device=device,
            )

            # Afficher seulement le déroulement du vrai mot masqué
            if candidate_id == masked_token_id:
                print("\n" + "-" * 60)
                print("Test du vrai candidat :", masked_token)
                print("ID du candidat :", candidate_id)
                print("Tenseur du candidat :", current_token)
                print(
                    "Score avant le contexte droit :",
                    vocab_probs[candidate_id]["log-prob"]
                )

            # ------------------------------------------------------
            # 4. Prédire tous les mots situés après le masque
            # ------------------------------------------------------
            for right_target_id in right_ids:

                logits, hidden = model.forward_evaluate(
                    current_token,
                    hidden
                )

                next_token_logits = logits[0, -1, :]

                next_token_log_probs = F.log_softmax(
                    next_token_logits,
                    dim=-1,
                )

                added_log_prob = next_token_log_probs[
                    right_target_id
                ].item()

                vocab_probs[candidate_id]["log-prob"] += (
                    added_log_prob
                )

                if candidate_id == masked_token_id:
                    print("\nMot fourni au GRU :", current_token.item())
                    print("Vrai prochain mot attendu :", right_target_id)
                    print("Shape des logits :", logits.shape)
                    print("Shape de hidden :", hidden.shape)
                    print(
                        "Log-probabilité ajoutée :",
                        added_log_prob
                    )
                    print(
                        "Score cumulé :",
                        vocab_probs[candidate_id]["log-prob"]
                    )

                # Introduire le vrai mot observé dans la phrase
                current_token = torch.tensor(
                    [[right_target_id]],
                    dtype=torch.long,
                    device=device,
                )

    # ---------------------------------------------------------
    # Trier les candidats par score décroissant
    # ---------------------------------------------------------
    top_k_probs = sorted(
        vocab_probs.items(),
        key=lambda item: item[1]["log-prob"],
        reverse=True,
    )[:top_k]

    print("\n" + "=" * 60)
    print(f"Mot attendu : {masked_token}")
    print(f"Top-{top_k} final :")

    rank = 1

    for token_id, info in top_k_probs:
        print(
            f"{rank}. {info['token']} "
            f"| id = {token_id} "
            f"| score = {info['log-prob']:.4f}"
        )
        rank += 1

    # Vérifier si le vrai mot est dans le Top-K
    for _, information in top_k_probs:
        if information["token"] == masked_token:
            print(
                f"\nLe mot '{masked_token}' "
                f"est dans le Top-{top_k}."
            )
            return True

    print(
        f"\nLe mot '{masked_token}' "
        f"n'est pas dans le Top-{top_k}."
    )

    return False

In [7]:
import torch

line = "7 <s> <s> <s> on suffo #quait , les chevel #ures s ' alourd #issaient sur les têtes en sueur . </s> </s> </s>"

config_file = "configs/config_1.json"
model_path = "best_models_strategy_1/GRU_h256_embDim32.pt"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Charger le modèle et ses poids
model = GRU.load(
    model_path=model_path,
    json_path=config_file
)

# Déplacer le modèle sur CPU ou GPU
model = model.to(device)
model.eval()

result = process_sentence(
    line=line,
    vocab=model.vocab,
    device=device,
    model=model,
    top_k=10
)

print("Résultat :", result)

vocab size (depuis fichier global) = 12001 emb_dim = 32
Taille du vocabulaire : 12001
Indice de <pad> : 12000
Indice de <unk> : 0
Taille embeddings : torch.Size([12001, 32])

Ligne reçue : 7 <s> <s> <s> on suffo #quait , les chevel #ures s ' alourd #issaient sur les têtes en sueur . </s> </s> </s>
line_parts : ['7', '<s>', '<s>', '<s>', 'on', 'suffo', '#quait', ',', 'les', 'chevel', '#ures', 's', "'", 'alourd', '#issaient', 'sur', 'les', 'têtes', 'en', 'sueur', '.', '</s>', '</s>', '</s>']

Position du mot masqué : 7
Mot masqué : les
ID du mot masqué : 249

Contexte gauche : ['<s>', '<s>', '<s>', 'on', 'suffo', '#quait']
Contexte droit : ['chevel', '#ures', 's', "'", 'alourd', '#issaient', 'sur', 'les', 'têtes', 'en', 'sueur', '.', '</s>', '</s>', '</s>']

IDs du contexte gauche : [1, 1, 1, 322, 7170, 1796]
IDs du contexte droit : [5657, 871, 53, 6, 11886, 1758, 317, 249, 3188, 245, 4402, 13, 2, 2, 2]

Tenseur du contexte gauche : tensor([[   1,    1,    1,  322, 7170, 1796]])
Shape du

## Running Tests 
Cette section sert à décortiquer la fonction forward() de notre GRU. 

In [9]:
config_file = "configs/config_1_test.json"
hp = HP_GRU(config_file)
model = GRU(hp)

train_file = model.hp.train_filename
val_file = model.hp.val_filename
vocab_filename = model.hp.vocab_filename

with open(vocab_filename, 'r') as f:
    data = json.load(f)

tok = data["model"]["vocab"]

train_batches = pad_sequences(
        train_file,
        tok,
        batch_size=model.hp.minibatch,
        max_diff=5,
        pad_token="<pad>",
        unk_token="<unk>",
        shuffle_batches=True
)
val_batches = pad_sequences(
        val_file,
        tok,
        batch_size=model.hp.minibatch,
        max_diff=5,
        pad_token="<pad>",
        unk_token="<unk>",
        shuffle_batches=True
)

train_loader = DataLoader(
    train_batches,
    batch_size=model.hp.minibatch,
    shuffle=True,
    collate_fn=lambda batch: batch[0]
)

val_loader = DataLoader(
    val_batches,
    batch_size=model.hp.minibatch,
    shuffle=True,
    collate_fn=lambda batch: batch[0]
)

vocab size (depuis fichier global) = 12001 emb_dim = 32
Taille du vocabulaire : 12001
Indice de <pad> : 12000
Indice de <unk> : 0
Taille embeddings : torch.Size([12001, 32])


In [10]:
exe_time, best_metrics = model.fit(train_loader, val_loader, 5, 1, lr=1e-3, patience=1)

torch.Size([3, 6])
Vraies longueurs des préfixes :  tensor([3, 3, 6])
Shape x packed (prefixe packed) : torch.Size([12, 32])
Shape output_packed : torch.Size([12, 64])
Shape h_n : torch.Size([1, 3, 64])
Shape output : torch.Size([3, 6, 64])
Shape logits : torch.Size([3, 6, 12001])
--------------------------------------------------------------
torch.Size([2, 51])
Vraies longueurs des préfixes :  tensor([47, 51])
Shape x packed (prefixe packed) : torch.Size([98, 32])
Shape output_packed : torch.Size([98, 64])
Shape h_n : torch.Size([1, 2, 64])
Shape output : torch.Size([2, 51, 64])
Shape logits : torch.Size([2, 51, 12001])
--------------------------------------------------------------
torch.Size([3, 11])
Vraies longueurs des préfixes :  tensor([10, 10, 11])
Shape x packed (prefixe packed) : torch.Size([31, 32])
Shape output_packed : torch.Size([31, 64])
Shape h_n : torch.Size([1, 3, 64])
Shape output : torch.Size([3, 11, 64])
Shape logits : torch.Size([3, 11, 12001])
--------------------